In [1]:
# Cell 1: Imports & Dependencies
import numpy as np
import heapq, time, random, itertools
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from scipy.optimize import linprog
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")

✓ All imports successful


In [2]:
# Cell 2: Core Data Structures

@dataclass
class TOPInstance:
    n: int                  # number of nodes (incl. depot start/end)
    m: int                  # number of vehicles
    T_max: float            # time budget per vehicle
    profits: np.ndarray     # shape (n,)  — depot nodes have profit=0
    dist: np.ndarray        # shape (n,n) — travel time/distance matrix
    depot_start: int = 0
    depot_end:   int = -1   # resolved to n-1 at runtime

    def end(self):
        return self.n - 1 if self.depot_end == -1 else self.depot_end

@dataclass
class Route:
    nodes:        List[int]
    profit:       float
    duration:     float
    reduced_cost: float = 0.0

@dataclass(order=False)
class BPNode:
    depth:       int
    lp_bound:    float
    forced_in:   List[Tuple[int,int]] = field(default_factory=list)
    forced_out:  List[Tuple[int,int]] = field(default_factory=list)
    def __lt__(self, other):          # max-heap on lp_bound
        return self.lp_bound > other.lp_bound

print("✓ Data structures defined")

✓ Data structures defined


In [3]:
# Cell 3: Random Instance Generator

def make_instance(n: int = 10, m: int = 2, T_max: float = 15.0,
                  seed: int = 42) -> TOPInstance:
    """
    Generate a random Euclidean TOP instance.
    Node 0        = depot start  (profit = 0)
    Node n-1      = depot end    (profit = 0)
    Nodes 1..n-2  = customers    (profit ~ Uniform[1,9])
    """
    rng = np.random.default_rng(seed)
    coords = rng.uniform(0, 10, (n, 2))
    dist   = np.linalg.norm(coords[:, None] - coords[None, :], axis=-1)
    profits        = np.zeros(n)
    profits[1:-1]  = rng.integers(1, 10, n - 2).astype(float)
    return TOPInstance(n=n, m=m, T_max=T_max, profits=profits, dist=dist)

# Quick sanity check
inst_test = make_instance(n=8, m=2, T_max=12.0, seed=0)
print(f"✓ Instance: n={inst_test.n}, m={inst_test.m}, T_max={inst_test.T_max}")
print(f"  Profits : {inst_test.profits}")
print(f"  Dist[0] : {inst_test.dist[0].round(2)}")

✓ Instance: n=8, m=2, T_max=12.0
  Profits : [0. 1. 8. 1. 5. 1. 3. 0.]
  Dist[0] : [0.   6.48 6.67 4.61 6.72 3.21 3.23 1.32]


In [4]:
# Cell 4: Greedy Initialisation (warm-start for RMP)

def initial_routes(inst: TOPInstance) -> List[Route]:
    """
    Each vehicle greedily picks the highest-profit unvisited node
    reachable within T_max: depot → node → depot.
    """
    s, e = inst.depot_start, inst.end()
    used, routes = set(), []
    for _ in range(inst.m):
        best_node, best_p = -1, -1.0
        for v in range(1, inst.n - 1):
            if v in used:
                continue
            if inst.dist[s, v] + inst.dist[v, e] <= inst.T_max:
                if inst.profits[v] > best_p:
                    best_p, best_node = inst.profits[v], v
        if best_node >= 0:
            used.add(best_node)
            dur = inst.dist[s, best_node] + inst.dist[best_node, e]
            routes.append(Route([s, best_node, e], best_p, dur))
        else:
            routes.append(Route([s, e], 0.0, inst.dist[s, e]))
    return routes

r0 = initial_routes(inst_test)
print("✓ Initial routes:")
for k, r in enumerate(r0):
    print(f"  Vehicle {k+1}: {r.nodes}  profit={r.profit}  dur={r.duration:.2f}")

✓ Initial routes:
  Vehicle 1: [0, 6, 7]  profit=3.0  dur=5.14
  Vehicle 2: [0, 3, 7]  profit=1.0  dur=10.28


In [5]:
# Cell 5: Restricted Master Problem — LP Relaxation
#
#   max  Σ_{r∈R} p_r · x_r
#   s.t. Σ_{r: i∈r} x_r ≤ 1   ∀i ∈ N  (visit-once)
#        0 ≤ x_r ≤ 1            ∀r ∈ R  (LP relaxation)

def solve_rmp(inst: TOPInstance,
              routes: List[Route]) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Solve LP relaxation of set-packing master via HiGHS (scipy).
    Returns  (x_vals, dual_prices[0..n-1], lp_obj)
    dual_prices[i] = π_i = marginal value of visiting node i.
    """
    interior  = list(range(1, inst.n - 1))   # non-depot nodes
    n_r       = len(routes)
    n_cstr    = len(interior)

    # Coverage matrix  A[i, r] = 1  iff  interior[i] ∈ routes[r].nodes
    A = np.zeros((n_cstr, n_r))
    for r, route in enumerate(routes):
        node_set = set(route.nodes)
        for i, node in enumerate(interior):
            if node in node_set:
                A[i, r] = 1.0

    c      = -np.array([r.profit for r in routes])   # minimise -profit
    bounds = [(0.0, 1.0)] * n_r

    res = linprog(c, A_ub=A, b_ub=np.ones(n_cstr), bounds=bounds, method='highs')

    if res.status != 0:
        return np.zeros(n_r), np.zeros(inst.n), 0.0

    x_vals = res.x
    dual_prices = np.zeros(inst.n)
    if res.ineqlin is not None:
        for i, node in enumerate(interior):
            dual_prices[node] = max(0.0, -res.ineqlin.marginals[i])

    return x_vals, dual_prices, float(-res.fun)

x0, pi0, obj0 = solve_rmp(inst_test, r0)
print(f"✓ RMP solved:  LP obj = {obj0:.4f}")
print(f"  x_vals     : {x0.round(3)}")
print(f"  dual_prices: {pi0.round(3)}")

✓ RMP solved:  LP obj = 4.0000
  x_vals     : [1. 1.]
  dual_prices: [0. 0. 0. 0. 0. 0. 0. 0.]


In [6]:
# Cell 6: Pricing Subproblem — Label-Setting DP
#
#   Reduced profit: p̄_i = p_i − π_i
#   Find route r with max  c̄_r = Σ_{i∈r} p̄_i  subject to T_max
#   Add route iff  c̄* > 0  (improving column exists)

def pricing_dp(inst: TOPInstance,
               dual_prices: np.ndarray,
               forced_in:  List[Tuple[int,int]] = [],
               forced_out: List[Tuple[int,int]] = []) -> Optional[Route]:
    """
    Label-setting DP for the elementary shortest path problem
    with resource constraint (time ≤ T_max).

    Label = (time_used, reduced_profit, path_list)
    Dominance: label A dominates B if  A.time ≤ B.time  AND  A.rprofit ≥ B.rprofit
               AND  A visits a subset of nodes of B  (relaxed: same visited set).
    """
    n   = inst.n
    s   = inst.depot_start
    e   = inst.end()
    T   = inst.T_max
    p_b = inst.profits - dual_prices          # reduced profits

    # labels[v] = list of (time, reduced_profit, path)
    labels = {v: [] for v in range(n)}
    labels[s] = [(0.0, 0.0, [s])]

    for _ in range(n - 1):
        next_labels = {v: [] for v in range(n)}
        for u, lbls in labels.items():
            for (t, rp, path) in lbls:
                path_set = set(path)
                for v in range(n):
                    if v in path_set:
                        continue
                    if (u, v) in forced_out:
                        continue
                    t2 = t + inst.dist[u, v]
                    if t2 + inst.dist[v, e] > T:
                        continue
                    next_labels[v].append((t2, rp + p_b[v], path + [v]))

        # Dominance pruning within each node
        for v in next_labels:
            pareto = []
            for (ta, ra, pa) in sorted(next_labels[v], key=lambda x: x[0]):
                dominated = False
                for (tb, rb, _) in pareto:
                    if tb <= ta and rb >= ra:
                        dominated = True
                        break
                if not dominated:
                    pareto.append((ta, ra, pa))
            labels[v] = pareto

    # Collect complete routes ending at depot_end
    candidates = []
    for (t, rp, path) in labels.get(e, []):
        full = path if path[-1] == e else path + [e]
        raw_profit   = sum(inst.profits[v] for v in full[1:-1])
        reduced_cost = sum(p_b[v]          for v in full[1:-1])
        candidates.append(Route(full, raw_profit, t, reduced_cost))

    if not candidates:
        return None
    best = max(candidates, key=lambda r: r.reduced_cost)
    return best if best.reduced_cost > 1e-6 else None

# Test pricing
new_col = pricing_dp(inst_test, pi0)
if new_col:
    print(f"✓ New column found: {new_col.nodes}  reduced_cost={new_col.reduced_cost:.4f}  profit={new_col.profit}")
else:
    print("✓ No improving column (LP already optimal with initial routes)")

✓ No improving column (LP already optimal with initial routes)


In [7]:
# Cell 7: Column Generation Subroutine

def column_generation(inst:       TOPInstance,
                      routes:     List[Route],
                      forced_in:  List[Tuple[int,int]] = [],
                      forced_out: List[Tuple[int,int]] = [],
                      max_iter:   int  = 100,
                      verbose:    bool = False):
    """
    while true:
        1. Solve RMP  →  get dual prices π
        2. For each vehicle k: solve pricing DP  (p̄_i = p_i - π_i)
        3. If c̄* > 0: add column
        4. Else: STOP  (LP optimal for current node)
    Returns (routes, dual_prices, lp_obj, history_of_lp_obj)
    """
    history = []
    for it in range(max_iter):
        x_vals, duals, lp_obj = solve_rmp(inst, routes)
        history.append(lp_obj)

        added = False
        for k in range(inst.m):
            col = pricing_dp(inst, duals, forced_in, forced_out)
            if col is not None:
                # Avoid exact duplicates
                existing = [tuple(r.nodes) for r in routes]
                if tuple(col.nodes) not in existing:
                    routes.append(col)
                    added = True

        if not added:
            if verbose:
                print(f"  CG converged at iteration {it+1:>3d}  |  LP = {lp_obj:.4f}  |  |R| = {len(routes)}")
            break

    return routes, duals, lp_obj, history

# Test on small instance
inst_cg = make_instance(n=10, m=2, T_max=15.0, seed=42)
routes_cg = initial_routes(inst_cg)
routes_cg, duals_cg, lp_cg, hist_cg = column_generation(inst_cg, routes_cg, verbose=True)
print(f"  Total columns in RMP : {len(routes_cg)}")
print(f"  CG iterations        : {len(hist_cg)}")

  CG converged at iteration   1  |  LP = 18.0000  |  |R| = 2
  Total columns in RMP : 2
  CG iterations        : 1


In [8]:
# Cell 8: Branch-and-Price — Main Solver

def is_integer_solution(x_vals: np.ndarray, tol: float = 1e-5) -> bool:
    return all(v < tol or v > 1.0 - tol for v in x_vals)

def extract_integer_solution(x_vals, routes):
    return [r for x, r in zip(x_vals, routes) if x > 0.5]

def ryan_foster_branch(x_vals, routes) -> Optional[Tuple[int,int]]:
    """
    Ryan-Foster branching: find pair (i,j) such that
    Σ_{r: i∈r, j∈r} x_r  is fractional → branch on together/apart.
    """
    pair_sum: Dict[Tuple[int,int], float] = {}
    for x, r in zip(x_vals, routes):
        interior = r.nodes[1:-1]
        for i, j in itertools.combinations(sorted(interior), 2):
            pair_sum[(i, j)] = pair_sum.get((i, j), 0.0) + x
    for pair, total in pair_sum.items():
        if 1e-5 < (total - round(total)) < 1.0 - 1e-5:
            return pair
    return None

def branch_and_price(inst:       TOPInstance,
                     time_limit: float = 120.0,
                     verbose:    bool  = True) -> Tuple[List[Route], float, dict]:
    """
    Branch-and-Price with:
      - Best-first search (priority queue on LP bound)
      - Column generation at every B&B node
      - Ryan-Foster branching on fractional node-pairs
      - LP-bound pruning
    """
    t_start        = time.time()
    best_profit    = 0.0
    best_routes: List[Route] = []
    nodes_explored = 0
    total_lp_solves = 0
    total_labels   = 0

    base_routes = initial_routes(inst)
    root = BPNode(depth=0, lp_bound=float('inf'))
    heap = [(0.0, 0, root)]   # (neg_bound, tie_break, node)
    tie  = 0

    while heap and (time.time() - t_start) < time_limit:
        neg_bound, _, bp_node = heapq.heappop(heap)
        nodes_explored += 1

        routes_here = list(base_routes)           # fresh copy each node
        routes_here, duals, lp_obj, hist = column_generation(
            inst, routes_here,
            bp_node.forced_in, bp_node.forced_out,
            verbose=False)
        total_lp_solves += len(hist)

        # Update LP bound on this node
        bp_node.lp_bound = lp_obj

        # ── Prune: LP bound cannot improve on best integer solution
        if lp_obj <= best_profit + 1e-5:
            if verbose:
                print(f"  Node {nodes_explored:>3d} | depth={bp_node.depth} | PRUNED  (LP={lp_obj:.2f} ≤ best={best_profit:.2f})")
            continue

        x_vals, _, _ = solve_rmp(inst, routes_here)
        total_lp_solves += 1

        if is_integer_solution(x_vals):
            # ── Integer feasible leaf
            sol    = extract_integer_solution(x_vals, routes_here)
            profit = sum(r.profit for r in sol)
            if verbose:
                status = "★ NEW BEST" if profit > best_profit else "  feasible"
                print(f"  Node {nodes_explored:>3d} | depth={bp_node.depth} | INTEGER  profit={profit:.1f}  {status}")
            if profit > best_profit:
                best_profit = profit
                best_routes = sol
        else:
            # ── Branch
            pair = ryan_foster_branch(x_vals, routes_here)
            if pair is not None:
                i, j = pair
                if verbose:
                    print(f"  Node {nodes_explored:>3d} | depth={bp_node.depth} | BRANCH   LP={lp_obj:.2f}  on ({i},{j})")
                for child_fi, child_fo in [
                        (bp_node.forced_in + [(i,j)], bp_node.forced_out),
                        (bp_node.forced_in,            bp_node.forced_out + [(i,j)])]:
                    child = BPNode(bp_node.depth + 1, lp_obj, child_fi, child_fo)
                    tie  += 1
                    heapq.heappush(heap, (-lp_obj, tie, child))

    elapsed = time.time() - t_start
    stats = {
        'nodes_explored': nodes_explored,
        'total_lp_solves': total_lp_solves,
        'cpu_time': round(elapsed, 4),
        'optimality': 'Optimal' if not heap else 'Time-limit'
    }
    return best_routes, best_profit, stats

print("✓ Branch-and-Price solver defined")

✓ Branch-and-Price solver defined


In [9]:
# Cell 9: Solution Quality & Performance Metrics

def gap_to_bks(bks: float, calc: float) -> float:
    """Gap(%) = 100 × (BKS − Calculated) / BKS"""
    return 0.0 if bks == 0 else 100.0 * (bks - calc) / bks

def gap_to_optimal(opt: float, calc: float) -> float:
    """Gap(%) = 100 × (Optimal − Calculated) / Optimal"""
    return 0.0 if opt == 0 else 100.0 * (opt - calc) / opt

def print_solution(routes, profit, stats, inst):
    print("=" * 55)
    print(f"  Optimal profit      : {profit:.1f}")
    print(f"  B&B nodes explored  : {stats['nodes_explored']}")
    print(f"  Total LP solves     : {stats['total_lp_solves']}")
    print(f"  CPU time            : {stats['cpu_time']} s")
    print(f"  Status              : {stats['optimality']}")
    print("-" * 55)
    for k, r in enumerate(routes):
        print(f"  Vehicle {k+1}: {r.nodes}")
        print(f"    profit={r.profit:.1f}   duration={r.duration:.3f}   (T_max={inst.T_max})")
    print("=" * 55)

print("✓ Metrics defined")

✓ Metrics defined


In [10]:
# Cell 10: Run B&P on a single instance

inst = make_instance(n=12, m=2, T_max=16.0, seed=7)

print(f"Instance: n={inst.n}, m={inst.m}, T_max={inst.T_max}")
print(f"Profits : {inst.profits}\n")

best_routes, best_profit, stats = branch_and_price(inst, time_limit=60, verbose=True)
print()
print_solution(best_routes, best_profit, stats, inst)

Instance: n=12, m=2, T_max=16.0
Profits : [0. 5. 1. 2. 5. 9. 5. 8. 9. 8. 6. 0.]

  Node   1 | depth=0 | INTEGER  profit=18.0  ★ NEW BEST

  Optimal profit      : 18.0
  B&B nodes explored  : 1
  Total LP solves     : 2
  CPU time            : 0.0058 s
  Status              : Optimal
-------------------------------------------------------
  Vehicle 1: [0, 5, 11]
    profit=9.0   duration=10.859   (T_max=16.0)
  Vehicle 2: [0, 8, 11]
    profit=9.0   duration=12.259   (T_max=16.0)


In [11]:
# Cell 11: Experiment 1 — Scalability with number of nodes

results_scale = []

configs = [
    (8,  2, 12.0),
    (10, 2, 14.0),
    (12, 2, 16.0),
    (15, 2, 18.0),
    (18, 3, 18.0),
    (20, 3, 20.0),
]

for (n, m, T) in configs:
    times_n = []
    profits_n = []
    lp_solves_n = []
    for seed in range(3):    # 3 seeds each
        inst = make_instance(n=n, m=m, T_max=T, seed=seed)
        rt, profit, stats = branch_and_price(inst, time_limit=30, verbose=False)
        times_n.append(stats['cpu_time'])
        profits_n.append(profit)
        lp_solves_n.append(stats['total_lp_solves'])

    results_scale.append({
        'n': n, 'm': m, 'T_max': T,
        'avg_profit':   round(np.mean(profits_n),  2),
        'avg_cpu':      round(np.mean(times_n),    3),
        'avg_lp_solves': round(np.mean(lp_solves_n), 1),
        'std_profit':   round(np.std(profits_n),   2),
    })
    print(f"n={n:>2d} m={m} T={T:.0f}  | profit={np.mean(profits_n):.1f}  cpu={np.mean(times_n):.3f}s  lp_solves={np.mean(lp_solves_n):.0f}")

df_scale = pd.DataFrame(results_scale)
print("\n", df_scale.to_string(index=False))

n= 8 m=2 T=12  | profit=13.0  cpu=0.005s  lp_solves=2
n=10 m=2 T=14  | profit=15.3  cpu=0.004s  lp_solves=2
n=12 m=2 T=16  | profit=18.0  cpu=0.007s  lp_solves=2
n=15 m=2 T=18  | profit=16.3  cpu=0.011s  lp_solves=2
n=18 m=3 T=18  | profit=24.7  cpu=0.015s  lp_solves=2
n=20 m=3 T=20  | profit=24.0  cpu=0.037s  lp_solves=2

  n  m  T_max  avg_profit  avg_cpu  avg_lp_solves  std_profit
 8  2   12.0       13.00    0.005            2.0        6.38
10  2   14.0       15.33    0.004            2.0        1.70
12  2   16.0       18.00    0.007            2.0        0.00
15  2   18.0       16.33    0.011            2.0        0.47
18  3   18.0       24.67    0.015            2.0        0.47
20  3   20.0       24.00    0.037            2.0        0.82


In [12]:
# Cell 12: Experiment 2 — Effect of number of vehicles

results_m = []

for m in [1, 2, 3, 4]:
    profits_m, times_m, lps_m = [], [], []
    for seed in range(5):
        inst = make_instance(n=12, m=m, T_max=16.0, seed=seed)
        rt, profit, stats = branch_and_price(inst, time_limit=30, verbose=False)
        profits_m.append(profit)
        times_m.append(stats['cpu_time'])
        lps_m.append(stats['total_lp_solves'])

    results_m.append({
        'm': m,
        'avg_profit':    round(np.mean(profits_m), 2),
        'std_profit':    round(np.std(profits_m),  2),
        'avg_cpu':       round(np.mean(times_m),   3),
        'avg_lp_solves': round(np.mean(lps_m),     1),
    })
    print(f"m={m} | avg_profit={np.mean(profits_m):.1f}  std={np.std(profits_m):.2f}  cpu={np.mean(times_m):.3f}s")

df_m = pd.DataFrame(results_m)
print("\n", df_m.to_string(index=False))

m=1 | avg_profit=9.0  std=0.00  cpu=0.008s
m=2 | avg_profit=17.4  std=0.80  cpu=0.008s
m=3 | avg_profit=25.0  std=1.67  cpu=0.009s
m=4 | avg_profit=31.4  std=2.15  cpu=0.011s

  m  avg_profit  std_profit  avg_cpu  avg_lp_solves
 1         9.0        0.00    0.008            2.0
 2        17.4        0.80    0.008            2.0
 3        25.0        1.67    0.009            2.0
 4        31.4        2.15    0.011            2.0


In [13]:
# Cell 13: Experiment 3 — Gap to LP bound across instances

gap_results = []

for seed in range(10):
    inst = make_instance(n=10, m=2, T_max=15.0, seed=seed)

    # Root LP bound (pure CG, no branching)
    r0 = initial_routes(inst)
    r0, duals, lp_bound, hist = column_generation(inst, r0, verbose=False)
    _, _, lp_obj = solve_rmp(inst, r0)

    # Full B&P integer solution
    rt, profit, stats = branch_and_price(inst, time_limit=30, verbose=False)

    gap_bks   = gap_to_bks(lp_obj, profit)          # gap: LP vs IP
    gap_opt   = gap_to_optimal(profit, profit)       # = 0 (B&P is optimal)

    gap_results.append({
        'seed': seed,
        'lp_bound': round(lp_obj, 3),
        'ip_profit': profit,
        'gap_lp_ip_%': round(gap_bks, 2),
        'cg_iters': len(hist),
        'cpu_s': stats['cpu_time'],
    })

df_gap = pd.DataFrame(gap_results)
print(df_gap.to_string(index=False))
print(f"\nAvg LP-IP gap : {df_gap['gap_lp_ip_%'].mean():.2f}%")
print(f"Avg CG iters  : {df_gap['cg_iters'].mean():.1f}")
print(f"Avg CPU time  : {df_gap['cpu_s'].mean():.3f}s")

 seed  lp_bound  ip_profit  gap_lp_ip_%  cg_iters  cpu_s
    0      13.0       13.0          0.0         1 0.0063
    1      16.0       16.0          0.0         1 0.0070
    2      17.0       17.0          0.0         1 0.0075
    3      18.0       18.0          0.0         1 0.0076
    4      16.0       16.0          0.0         1 0.0070
    5      18.0       18.0          0.0         1 0.0064
    6      13.0       13.0          0.0         1 0.0063
    7      16.0       16.0          0.0         1 0.0058
    8      17.0       17.0          0.0         1 0.0039
    9      18.0       18.0          0.0         1 0.0035

Avg LP-IP gap : 0.00%
Avg CG iters  : 1.0
Avg CPU time  : 0.006s
